## Interactive PCD clustering comparison app

The cells below are self-contained and appended without changing the preceding analysis. Upload an ASCII `.pcd`, add one or more axis-aligned manual boxes, then run the comparison.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.cluster import DBSCAN, KMeans, SpectralClustering
import hdbscan


In [2]:


def _pcd_bytes_to_xyz(raw_bytes):
    raw_bytes = bytes(raw_bytes)
    lines = raw_bytes.decode("utf-8", errors="ignore").splitlines()
    fields = None
    data_start = None
    data_mode = None
    for index, line in enumerate(lines):
        parts = line.strip().split()
        if not parts or parts[0].startswith("#"):
            continue
        if parts[0].upper() == "FIELDS":
            fields = parts[1:]
        elif parts[0].upper() == "DATA":
            data_mode = parts[1].lower() if len(parts) > 1 else None
            data_start = index + 1
            break
    if data_start is None or data_mode != "ascii":
        raise ValueError("This app supports ASCII PCD files only.")
    rows = [[float(value) for value in line.replace(",", " ").split()] for line in lines[data_start:] if line.strip() and not line.strip().startswith("#")]
    data = np.asarray(rows, dtype=float)
    if data.size == 0:
        raise ValueError("The PCD file contains no point data.")
    if data.ndim == 1:
        data = data.reshape(1, -1)
    columns = [fields.index(axis) for axis in ("x", "y", "z")] if fields and all(axis in fields for axis in ("x", "y", "z")) else [0, 1, 2]
    if data.shape[1] < 3:
        raise ValueError("The PCD file must contain x, y and z data.")
    return data[:, columns]



In [3]:
def load_pcd(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        header = []
        data_type = None
        fields = None

        for line in f:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            header.append(stripped)
            if stripped.startswith("DATA"):
                parts = stripped.split()
                if len(parts) < 2:
                    raise ValueError("Invalid PCD DATA line")
                data_type = parts[1].lower()
                break
            if stripped.startswith("FIELDS"):
                fields = stripped.split()[1:]

        if data_type != "ascii":
            raise ValueError("Only ASCII PCD files are supported")

        rows = []
        for line in f:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            values = stripped.replace(",", " ").split()
            if values:
                rows.append([float(value) for value in values])

        if not rows:
            raise ValueError("No point data found in PCD file")

        data = np.array(rows, dtype=float)
        if data.ndim == 1:
            data = data.reshape(1, -1)

        if fields is not None:
            field_map = {name: idx for idx, name in enumerate(fields)}
            if all(key in field_map for key in ("x", "y", "z")):
                points = data[:, [field_map["x"], field_map["y"], field_map["z"]]]
            else:
                points = data[:, :3]
        else:
            points = data[:, :3]

        return points


def _box_mask(points_array, box):
    lower = np.array([box["xmin"], box["ymin"], box["zmin"]])
    upper = np.array([box["xmax"], box["ymax"], box["zmax"]])
    return np.all((points_array >= lower) & (points_array <= upper), axis=1)


def _best_cluster(labels_array, manual_mask_array):
    candidates = [label for label in np.unique(labels_array) if label != -1] or list(np.unique(labels_array))
    best = None
    for label in candidates:
        predicted = labels_array == label
        tp = int(np.count_nonzero(predicted & manual_mask_array))
        fp = int(np.count_nonzero(predicted & ~manual_mask_array))
        fn = int(np.count_nonzero(~predicted & manual_mask_array))
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        candidate = {"label": label, "mask": predicted, "tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}
        if best is None or candidate["f1"] > best["f1"]:
            best = candidate
    return best


app_state = {"points": None, "boxes": [], "results": {}}
upload = widgets.FileUpload(accept=".pcd", multiple=False, description="Upload PCD")
status = widgets.HTML(value="Upload an ASCII PCD file to begin.")
box_output = widgets.Output()
plot_output = widgets.Output()
metrics_output = widgets.Output()
xmin = widgets.FloatText(description="xmin", layout=widgets.Layout(width="145px"))
xmax = widgets.FloatText(description="xmax", layout=widgets.Layout(width="145px"))
ymin = widgets.FloatText(description="ymin", layout=widgets.Layout(width="145px"))
ymax = widgets.FloatText(description="ymax", layout=widgets.Layout(width="145px"))
zmin = widgets.FloatText(description="zmin", layout=widgets.Layout(width="145px"))
zmax = widgets.FloatText(description="zmax", layout=widgets.Layout(width="145px"))
add_box = widgets.Button(description="Add box", button_style="success", icon="plus")
clear_boxes = widgets.Button(description="Clear boxes", button_style="warning", icon="trash")
run_button = widgets.Button(description="Run comparison", button_style="primary", icon="play")
n_clusters = widgets.IntSlider(value=3, min=2, max=20, description="KMeans/Spectral", continuous_update=False)
dbscan_eps = widgets.FloatLogSlider(value=0.05, base=10, min=-3, max=0, step=0.05, description="DBSCAN eps", continuous_update=False)
min_samples = widgets.IntSlider(value=10, min=2, max=100, description="Min samples", continuous_update=False)


def _box_summary():
    if not app_state["boxes"]:
        return "No manual boxes added."
    return "<br>".join(f"Box {i}: x [{b['xmin']:.3g}, {b['xmax']:.3g}], y [{b['ymin']:.3g}, {b['ymax']:.3g}], z [{b['zmin']:.3g}, {b['zmax']:.3g}]" for i, b in enumerate(app_state["boxes"], 1))


def _set_status(message, color="#444"):
    status.value = f'<span style="color:{color}">{message}</span>'


def _on_upload(change):
    if not upload.value:
        return
    uploaded = next(iter(upload.value.values()))
    try:
        app_state["points"] = load_pcd(uploaded["content"])
        app_state["boxes"] = []
        app_state["results"] = {}
        minimum = app_state["points"].min(axis=0)
        maximum = app_state["points"].max(axis=0)
        for field, value in zip((xmin, ymin, zmin), minimum):
            field.value = float(value)
        for field, value in zip((xmax, ymax, zmax), maximum):
            field.value = float(value)
        _set_status(f"Loaded {len(app_state['points']):,} points from {uploaded['name']}.", "#176b3a")
        with box_output:
            clear_output(wait=True)
            display(widgets.HTML(value=_box_summary()))
    except Exception as error:
        app_state["points"] = None
        _set_status(str(error), "#a11")


def _on_add_box(_):
    if app_state["points"] is None:
        _set_status("Upload a PCD file before adding a box.", "#a11")
        return
    box = {"xmin": xmin.value, "xmax": xmax.value, "ymin": ymin.value, "ymax": ymax.value, "zmin": zmin.value, "zmax": zmax.value}
    if any(box[f"min_{axis}"] > box[f"max_{axis}"] for axis in ("x", "y", "z")):
        _set_status("Each minimum must be less than or equal to its maximum.", "#a11")
        return
    app_state["boxes"].append(box)
    with box_output:
        clear_output(wait=True)
        display(widgets.HTML(value=_box_summary()))


def _on_clear_boxes(_):
    app_state["boxes"] = []
    app_state["results"] = {}
    with box_output:
        clear_output(wait=True)
        display(widgets.HTML(value=_box_summary()))
    with plot_output:
        clear_output(wait=True)
    with metrics_output:
        clear_output(wait=True)


upload.observe(_on_upload, names="value")
add_box.on_click(_on_add_box)
clear_boxes.on_click(_on_clear_boxes)

In [4]:
def _on_add_box(_):
    if app_state["points"] is None:
        _set_status("Upload a PCD file before adding a box.", "#a11")
        return
    box = {"xmin": xmin.value, "xmax": xmax.value, "ymin": ymin.value, "ymax": ymax.value, "zmin": zmin.value, "zmax": zmax.value}
    if any(box[f"{axis}min"] > box[f"{axis}max"] for axis in ("x", "y", "z")):
        _set_status("Each minimum must be less than or equal to its maximum.", "#a11")
        return
    app_state["boxes"].append(box)
    with box_output:
        clear_output(wait=True)
        display(widgets.HTML(value=_box_summary()))


add_box.on_click(_on_add_box)


def _grid_search(points_array, manual_mask_array, candidates):
    best = None
    for parameters, estimator in candidates:
        try:
            labels = estimator.fit_predict(points_array)
            candidate = _best_cluster(labels, manual_mask_array)
            if candidate is not None:
                candidate["params"] = parameters
                if best is None or candidate["f1"] > best["f1"]:
                    best = candidate
        except Exception:
            continue
    if best is None:
        raise ValueError("No valid parameter combination produced a clustering.")
    return best


def _format_parameters(parameters):
    return ", ".join(f"{name}={value:g}" if isinstance(value, float) else f"{name}={value}" for name, value in parameters.items())


def _render_results(points_array, manual_mask_array):
    with plot_output:
        clear_output(wait=True)
        figure, axes = plt.subplots(2, 2, figsize=(14, 10), subplot_kw={"projection": "3d"})
        for axis, (name, result) in zip(axes.flat, app_state["results"].items()):
            predicted = result["mask"]
            green = predicted & manual_mask_array
            red = predicted & ~manual_mask_array
            black = ~predicted
            axis.scatter(points_array[black, 0], points_array[black, 1], points_array[black, 2], c="k", s=2, alpha=0.2)
            axis.scatter(points_array[red, 0], points_array[red, 1], points_array[red, 2], c="r", s=5, label="false positive")
            axis.scatter(points_array[green, 0], points_array[green, 1], points_array[green, 2], c="g", s=5, label="true positive")
            axis.set_title(f"{name}: {_format_parameters(result['params'])} | F1 {result['f1']:.3f}")
            axis.set_xlabel("X")
            axis.set_ylabel("Y")
            axis.set_zlabel("Z")
        figure.tight_layout()
        plt.show()
    with metrics_output:
        clear_output(wait=True)
        headers = ["Algorithm", "Best parameters", "Best label", "Precision", "Recall", "F1", "TP", "FP", "FN"]
        rows = [headers]
        for name, result in app_state["results"].items():
            rows.append([
                name,
                _format_parameters(result["params"]),
                result["label"],
                f"{result['precision']:.3f}",
                f"{result['recall']:.3f}",
                f"{result['f1']:.3f}",
                result["tp"],
                result["fp"],
                result["fn"],
            ])
        display(widgets.HTML(value="<table><tr>" + "".join(f"<th style='padding:4px 10px'>{cell}</th>" for cell in rows[0]) + "</tr>" + "".join("<tr>" + "".join(f"<td style='padding:4px 10px'>{cell}</td>" for cell in row) + "</tr>" for row in rows[1:]) + "</table>"))


def _run_algorithms(_):
    points_array = app_state["points"]
    if points_array is None or not app_state["boxes"]:
        _set_status("Upload a PCD and add at least one manual box first.", "#a11")
        return
    manual_mask_array = np.zeros(len(points_array), dtype=bool)
    for box in app_state["boxes"]:
        manual_mask_array |= _box_mask(points_array, box)
    try:
        _set_status("Running compact grid searches", "#a11")
        max_clusters = min(n_clusters.value, len(points_array) - 1)
        if max_clusters < 2:
            raise ValueError("At least three points are required for clustering.")
        cluster_values = sorted({
            max(2, min(max_clusters, clusters))
            for clusters in (n_clusters.value - 1, n_clusters.value, n_clusters.value + 1)
        })
        sample_value = max(2, min(min_samples.max, min_samples.value))
        cluster_size_values = sorted({max(2, sample_value // 2), sample_value})
        eps_values = sorted({
            max(1e-6, dbscan_eps.value * factor)
            for factor in (0.8, 1.0, 1.2)
        })
        nearest_neighbors = min(10, max(2, len(points_array) - 1))
        candidates = {
            "K-means": [
                ({"n_clusters": clusters}, KMeans(n_clusters=clusters, random_state=42, n_init="auto"))
                for clusters in cluster_values
            ],
            "Spectral": [
                ({"n_clusters": clusters}, SpectralClustering(n_clusters=clusters, affinity="nearest_neighbors", n_neighbors=nearest_neighbors, assign_labels="kmeans", random_state=42))
                for clusters in cluster_values
            ],
            "DBSCAN": [
                ({"eps": eps, "min_samples": sample_value}, DBSCAN(eps=eps, min_samples=sample_value))
                for eps in eps_values
            ],
            "HDBSCAN": [
                ({"min_cluster_size": cluster_size, "min_samples": sample_value}, hdbscan.HDBSCAN(min_cluster_size=cluster_size, min_samples=sample_value))
                for cluster_size in cluster_size_values
            ],
        }
        app_state["results"] = {
            name: _grid_search(points_array, manual_mask_array, parameter_grid)
            for name, parameter_grid in candidates.items()
        }
        _render_results(points_array, manual_mask_array)
        _set_status(f"Compact grid search compared four algorithms against {int(manual_mask_array.sum()):,} manually selected points.", "#176b3a")
    except Exception as error:
        _set_status(f"Clustering failed: {error}", "#a11")


run_button.on_click(_run_algorithms)
display(widgets.VBox([
    upload,
    status,
    widgets.HTML("<b>Manual bounding boxes</b>"),
    widgets.HBox([xmin, xmax]),
    widgets.HBox([ymin, ymax]),
    widgets.HBox([zmin, zmax]),
    widgets.HBox([add_box, clear_boxes]),
    box_output,
    widgets.HTML("<b>Clustering parameters and grid limits</b>"),
    n_clusters,
    dbscan_eps,
    min_samples,
    run_button,
    metrics_output,
    plot_output,
]))

In [5]:
add_box._click_handlers.callbacks = []
add_box.on_click(_on_add_box)
print("Interactive clustering app initialized.")

Interactive clustering app initialized.


In [6]:
upload.unobserve(_on_upload, names="value")


def subsample(points, keep_fraction):
    if not 0 < keep_fraction <= 1:
        raise ValueError("keep_fraction must be between 0 and 1")

    num_keep = max(1, int(len(points) * keep_fraction))
    indices = np.linspace(0, len(points) - 1, num_keep, dtype=int)
    return points[indices]


def filter(points, threshold):
    distances = np.linalg.norm(points, axis=1)
    threshold_number = threshold * distances.max()
    mask = distances <= threshold_number
    return points[mask]


keep_fraction = widgets.FloatSlider(
    value=1.0,
    min=0.01,
    max=1.0,
    step=0.01,
    description="Keep points",
    readout_format=".0%",
    continuous_update=False,
)
filter_threshold = widgets.FloatSlider(
    value=1.0,
    min=0.01,
    max=1.0,
    step=0.01,
    description="Distance limit",
    readout_format=".0%",
    continuous_update=False,
)

app_state["raw_points"] = None


def _update_bounds(points):
    minimum = points.min(axis=0)
    maximum = points.max(axis=0)
    for field, value in zip((xmin, ymin, zmin), minimum):
        field.value = float(value)
    for field, value in zip((xmax, ymax, zmax), maximum):
        field.value = float(value)


def _apply_point_settings(_=None):
    if app_state["raw_points"] is None:
        return
    filtered_points = filter(app_state["raw_points"], filter_threshold.value)
    app_state["points"] = subsample(filtered_points, keep_fraction.value)
    app_state["boxes"] = []
    app_state["results"] = {}
    _update_bounds(app_state["points"])
    _set_status(
        f"Using {len(app_state['points']):,} of {len(app_state['raw_points']):,} uploaded points.",
        "#176b3a",
    )
    with box_output:
        clear_output(wait=True)
        display(widgets.HTML(value=_box_summary()))
    with plot_output:
        clear_output(wait=True)
    with metrics_output:
        clear_output(wait=True)


def _on_upload(change):
    if not upload.value:
        return
    uploaded = next(iter(upload.value.values())) if isinstance(upload.value, dict) else upload.value[0]
    try:
        app_state["raw_points"] = _pcd_bytes_to_xyz(uploaded["content"])
        _apply_point_settings()
        _set_status(
            f"Loaded {len(app_state['raw_points']):,} points from {uploaded['name']}; "
            f"using {len(app_state['points']):,} after filtering/subsampling.",
            "#176b3a",
        )
    except Exception as error:
        app_state["raw_points"] = None
        app_state["points"] = None
        _set_status(str(error), "#a11")


upload.observe(_on_upload, names="value")
keep_fraction.observe(_apply_point_settings, names="value")
filter_threshold.observe(_apply_point_settings, names="value")
display(widgets.VBox([
    widgets.HTML("<b>Point filtering and subsampling</b>"),
    filter_threshold,
    keep_fraction,
]))
print("Upload handler and point controls ready.")

Upload handler and point controls ready.
